# Bronze → Silver — Wikimedia Recent Changes

Parses the raw `event_json` string from Bronze into structured columns using
an explicit schema, classifies edits as bot vs human, and keeps the original
`event_json` alongside the parsed columns as a safety net.

## 1. Configuration

In [0]:
%run "../00-setup/00_config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    BooleanType,
)

SILVER_TABLE = f"{SILVER_SCHEMA}.wikipedia_edits"
SILVER_CHECKPOINT_PATH = f"{STORAGE_ROOT}/checkpoints/bronze_to_silver"
BRONZE_TABLE = f"{BRONZE_SCHEMA}.wikipedia_edits_stream"

print("Bronze table:", BRONZE_TABLE)
print("Silver table:", SILVER_TABLE)
print("Silver checkpoint:", SILVER_CHECKPOINT_PATH)

## 2. Explicit schema for the Wikimedia recentchange event

Only the fields we actually use downstream are listed here. Anything else in
the source event stays untouched inside `event_json`, which we carry forward
into Silver as well.

In [0]:
wikipedia_event_schema = StructType([
    StructField("id", LongType()),
    StructField("type", StringType()),
    StructField("namespace", LongType()),
    StructField("title", StringType()),
    StructField("title_url", StringType()),
    StructField("comment", StringType()),
    StructField("timestamp", LongType()),
    StructField("user", StringType()),
    StructField("bot", BooleanType()),
    StructField("minor", BooleanType()),
    StructField("patrolled", BooleanType()),
    StructField("server_name", StringType()),
    StructField("wiki", StringType()),
    StructField("meta", StructType([
        StructField("domain", StringType()),
        StructField("dt", StringType()),
    ])),
    StructField("length", StructType([
        StructField("old", LongType()),
        StructField("new", LongType()),
    ])),
])

## 3. Read Bronze incrementally

In [0]:
df_bronze_stream = spark.readStream.format("delta").table(BRONZE_TABLE)

## 4. Parse, flag parse failures, classify bot vs human, keep raw JSON as a safety net

`_parse_error` is `true` when `from_json` could not parse `event_json` at all
(malformed JSON) — as opposed to a field simply being absent from a
well-formed event. This distinguishes "broken record" from "legitimately
missing field" instead of letting both collapse into the same silent nulls.

In [0]:
df_parsed = (
    df_bronze_stream
    .withColumn("event", F.from_json(F.col("event_json"), wikipedia_event_schema))
    .withColumn("_parse_error", F.col("event").isNull())
)

df_silver = (
    df_parsed
    .select(
        F.col("event.id").alias("event_id"),
        F.col("event.type").alias("event_type"),
        F.col("event.namespace").alias("namespace"),
        F.col("event.title").alias("title"),
        F.col("event.title_url").alias("title_url"),
        F.col("event.comment").alias("comment"),
        F.col("event.timestamp").alias("event_timestamp_unix"),
        F.from_unixtime(F.col("event.timestamp")).cast("timestamp").alias("event_time"),
        F.to_timestamp(F.col("event.meta.dt")).alias("event_time_utc"),
        F.col("event.user").alias("user"),
        F.col("event.bot").alias("bot"),
        F.col("event.minor").alias("minor"),
        F.col("event.patrolled").alias("patrolled"),
        F.col("event.server_name").alias("server_name"),
        F.col("event.wiki").alias("wiki"),
        F.col("event.meta.domain").alias("domain"),
        F.col("event.length.old").alias("length_old"),
        F.col("event.length.new").alias("length_new"),
        F.col("_parse_error"),
        F.col("event_json"),  # raw JSON kept as a fallback for schema drift
        F.col("kafka_topic"),
        F.col("kafka_partition"),
        F.col("kafka_offset"),
        F.col("kafka_enqueued_at"),
        F.col("_ingested_at").alias("bronze_ingested_at"),
        F.col("_load_date").alias("bronze_load_date"),
    )
    .withColumn(
        "editor_type",
        F.when(F.col("bot") == True, "bot").otherwise("human"),
    )
    .withColumn("silver_ingested_at", F.current_timestamp())
    .withColumn("silver_load_date", F.current_date())
)

## 5. Write to Silver

In [0]:
query = (
    df_silver.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", SILVER_CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE)
)

query.awaitTermination()

print("Row count:", spark.table(SILVER_TABLE).count())

## 6. Sanity check

In [0]:
spark.table(SILVER_TABLE).printSchema()
display(spark.table(SILVER_TABLE).orderBy(F.col("silver_ingested_at").desc()).limit(5))

## 6b. Parse error rate

Rows where `_parse_error = true` had `event_json` that `from_json` could not
parse at all. This should normally be at or near zero — a nonzero count is
worth investigating (truncated JSON, encoding issue, etc.), separately from
the schema-evolution case where a *new field* is simply absent from parsed
columns but the JSON itself parses fine.

In [0]:
display(spark.table(SILVER_TABLE).groupBy("_parse_error").count())

## 7. Bot vs human split

In [0]:
display(spark.table(SILVER_TABLE).groupBy("editor_type").count())

## 8. Edits per project / domain (top 10)

In [0]:
display(
    spark.table(SILVER_TABLE)
    .groupBy("domain")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

## 9. Idempotency check — rerun with the same checkpoint

In [0]:
before_count = spark.table(SILVER_TABLE).count()
print(f"Row count BEFORE re-run: {before_count}")

In [0]:
query = (
    df_silver.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", SILVER_CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE)
)
query.awaitTermination()

after_count = spark.table(SILVER_TABLE).count()
print(f"Row count AFTER re-run: {after_count}")

if after_count == before_count:
    print("✅ IDEMPOTENCY CONFIRMED — no duplicate rows without new Bronze data")
else:
    print(f"ℹ️ Row count changed by {after_count - before_count} — expected if Bronze received new rows between runs")

In [0]:
display(
    spark.table(SILVER_TABLE)
    .groupBy("domain")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)
)

In [0]:
display(
    spark.table(SILVER_TABLE)
    .groupBy(F.window("event_time", "1 hour").alias("hour_window"))
    .count()
    .select(
        F.col("hour_window.start").alias("hour"),
        "count"
    )
    .orderBy("hour")
)

In [0]:
display(
    spark.table(SILVER_TABLE)
    .groupBy(F.window("event_time", "1 minute").alias("min_window"))
    .count()
    .select(
        F.col("min_window.start").alias("minute"),
        "count"
    )
    .orderBy("minute")
)

In [0]:
#duplication check
%sql
SELECT event_json, COUNT(*) AS cnt
FROM dbr_dev.wikimediademo_bronze.wikipedia_edits_stream
GROUP BY event_json
HAVING COUNT(*) > 1